In [ ]:
# # 根据机器人名称自动计算阈值（推荐）
# drg = DRG(grid, resolution=0.05, robot_name='burger')

# # 或手动覆盖
# drg = DRG(grid, resolution=0.05, merge_threshold=8)

# # 提取拓扑
# drg.extract()

In [11]:
import importlib.util
import matplotlib.pyplot as plt
import numpy as np

# 正确的文件路径
drg_path = '/home/chendawww/workspace/rl-navibot/src/perception/perception/slam/drg.py'
maps_path = '/home/chendawww/workspace/rl-navibot/src/perception/perception/slam/baselines/maps.py'

def load_class_from_path(filepath, class_name):
    spec = importlib.util.spec_from_file_location(class_name, filepath)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return getattr(module, class_name)

def load_func_from_path(filepath, func_name):
    spec = importlib.util.spec_from_file_location(func_name, filepath)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return getattr(module, func_name)

DRG = load_class_from_path(drg_path, 'DRG')
make_baseline_grid = load_func_from_path(maps_path, 'make_baseline_grid')

# 后续测试代码不变...
def main():
    print("生成 baseline 栅格图...")
    grid, W, H, res = make_baseline_grid()
    print(f"栅格尺寸: {W}x{H}, 分辨率: {res} m/pixel")

    print("初始化 DRG 并提取拓扑...")
    drg = DRG(grid, resolution=res)
    drg.extract()

    print(f"提取到 {len(drg.nodes)} 个节点，{len(drg.edges)} 条边")
    for n in drg.nodes[:5]:  # 打印前5个节点
        print(f"  {n['id']}: x={n['x']:.2f}, y={n['y']:.2f}, span=({n['span_x']:.2f}, {n['span_y']:.2f})")

    print("保存可视化图片到 drg_baseline.png ...")
    drg.visualize(save_path="./drg_baseline.png")
    print("完成！")

if __name__ == "__main__":
    main()

生成 baseline 栅格图...
栅格尺寸: 250x200, 分辨率: 0.1 m/pixel
初始化 DRG 并提取拓扑...
提取到 176 个节点，333 条边
  N_0: x=4.40, y=0.80, span=(6.00, 1.90)
  N_1: x=4.90, y=0.80, span=(6.00, 1.40)
  N_2: x=7.40, y=0.80, span=(6.00, 1.30)
  N_3: x=7.90, y=0.80, span=(6.00, 1.30)
  N_4: x=14.40, y=0.80, span=(6.00, 1.30)
保存可视化图片到 drg_baseline.png ...
完成！


In [8]:
import sys
import os

# 明确指向源码中的 perception 包
src_perception = '/home/chendawww/workspace/rl-navibot/src/perception'
if src_perception not in sys.path:
    sys.path.insert(0, src_perception)

# 可选：移除可能导致混淆的其他路径
sys.path = [p for p in sys.path if 'install' not in p and 'build' not in p]

# 验证导入路径
import perception
print(perception.__file__)  # 应该输出 src/perception/__init__.py

from perception.slam.drg import DRG
from perception.slam.baselines.maps import make_baseline_grid

print("✅ 导入成功，DRG 类来自:", DRG.__module__)

None


ModuleNotFoundError: No module named 'perception.slam'

In [3]:
import rclpy
from rclpy.node import Node
from nav2_msgs.msg import KeepoutFilterMask
from geometry_msgs.msg import Polygon, Point32

class KeepoutBoundaryNode(Node):
    def __init__(self):
        super().__init__('keepout_boundary_node')

        # --- 1. 定义你的“外圈”边界范围 ---
        # 假设你想限制在以(0,0)为中心，X从-6到6，Y从-6到6的范围内
        min_x, max_x = -6.0, 6.0
        min_y, max_y = -6.0, 6.0
        
        # 极其重要：墙的厚度！必须 > 2 * inflation_radius
        # 假设你的膨胀半径是 0.55，那墙厚至少设为 1.2
        thickness = 1.2 

        # --- 2. 拼装 4 条“墙壁”多边形 ---
        # 多边形是顺时针或逆时针一圈的点组成的
        polygons = []

        # 底边墙 (Y轴最小处的那条横墙)
        polygons.append(self._create_wall_rect(
            min_x, min_y, max_x, min_y + thickness))
        
        # 顶边墙 (Y轴最大处的那条横墙)
        polygons.append(self._create_wall_rect(
            min_x, max_y - thickness, max_x, max_y))
        
        # 左边墙 (X轴最小处的那条竖墙)
        polygons.append(self._create_wall_rect(
            min_x, min_y + thickness, min_x + thickness, max_y - thickness))
        
        # 右边墙 (X轴最大处的那条竖墙)
        polygons.append(self._create_wall_rect(
            max_x - thickness, min_y + thickness, max_x, max_y - thickness))

        # --- 3. 构造并发布 Mask 消息 ---
        self.publisher = self.create_publisher(KeepoutFilterMask, '/keepout_filter_mask', 10)
        
        mask_msg = KeepoutFilterMask()
        mask_msg.header.frame_id = 'map'
        mask_msg.filter_type = 0  # 0 代表 PROHIBIT (禁区)
        mask_msg.polygons = polygons

        # 稍微延迟一点发布，确保 Nav2 的 costmap 已经初始化完毕
        self.timer = self.create_timer(2.0, lambda: self._publish_mask(mask_msg))
        self.get_logger().info(f'已准备发送虚拟围墙，厚度: {thickness}m')

    def _create_wall_rect(self, x1, y1, x2, y2):
        """辅助函数：用左下角和右上角坐标生成一个矩形多边形"""
        poly = Polygon()
        poly.points = [
            Point32(x=x1, y=y1, z=0.0),
            Point32(x=x2, y=y1, z=0.0),
            Point32(x=x2, y=y2, z=0.0),
            Point32(x=x1, y=y2, z=0.0)
        ]
        return poly

    def _publish_mask(self, msg):
        self.publisher.publish(msg)
        self.get_logger().info('虚拟围墙已发布！ExploreLite 将被限制在框内。')
        self.timer.cancel() # 发一次就行了，不需要一直发

def main(args=None):
    rclpy.init(args=args)
    node = KeepoutBoundaryNode()
    rclpy.spin(node)
    node.destroy_node()
    rclpy.shutdown()


ImportError: cannot import name 'KeepoutFilterMask' from 'nav2_msgs.msg' (/opt/ros/humble/local/lib/python3.10/dist-packages/nav2_msgs/msg/__init__.py)

In [ ]:
import rclpy
from rclpy.node import Node
from nav2_msgs.msg import CostmapFilterInfo
from geometry_msgs.msg import PolygonStamped, Polygon, Point32

class KeepoutBoundaryNode(Node):
    def __init__(self):
        super().__init__('keepout_boundary_node')

        # --- 1. 定义你的“外圈”边界范围 ---
        # 假设限制在 X: [-6, 6], Y: [-6, 6]
        min_x, max_x = -6.0, 6.0
        min_y, max_y = -6.0, 6.0
        
        # 极其重要：墙的厚度！必须 > 2 * inflation_radius
        # 假设你的膨胀半径是 0.55，那墙厚至少设为 1.2
        thickness = 1.2 

        # --- 2. 拼装 4 条“墙壁”多边形 ---
        walls_polygon = [
            # 底边墙：从最左到最右
            self._create_wall_rect(min_x, min_y, max_x, min_y + thickness),               
            # 顶边墙：从最左到最右
            self._create_wall_rect(min_x, max_y - thickness, max_x, max_y),               
            # 左边墙：从最下到最上 (修正这里：改为 min_y 和 max_y)
            self._create_wall_rect(min_x, min_y, min_x + thickness, max_y), 
            # 右边墙：从最下到最上 (修正这里：改为 min_y 和 max_y)
            self._create_wall_rect(max_x - thickness, min_y, max_x, max_y)  
        ]

        # --- 3. 创建发布器 ---
        # 发布“说明书”
        self.info_pub = self.create_publisher(CostmapFilterInfo, '/keepout_filter_info', 10)
        # 发布“具体形状”
        self.poly_pub = self.create_publisher(PolygonStamped, '/keepout_polygons_topic', 10)

        # 保存数据供定时器使用
        self.walls_polygon = walls_polygon
        
        # 延迟 2 秒发布，确保 Nav2 的 global_costmap 已经完全启动并开始监听
        self.timer = self.create_timer(2.0, self.publish_boundaries)
        self.get_logger().info(f'已准备发送虚拟围墙，厚度: {thickness}m')

    def _create_wall_rect(self, x1, y1, x2, y2):
        """辅助函数：用左下角和右上角坐标生成一个矩形 Polygon"""
        poly = Polygon()
        poly.points = [
            Point32(x=x1, y=y1, z=0.0),
            Point32(x=x2, y=y1, z=0.0),
            Point32(x=x2, y=y2, z=0.0),
            Point32(x=x1, y=y2, z=0.0)
        ]
        return poly

    def publish_boundaries(self):
        # 第一步：发送说明书
        info_msg = CostmapFilterInfo()
        info_msg.type = 1  # 注意！0 是 THRESHOLD，1 才是 KEEPOUT (禁区)
        info_msg.base = "/keepout_polygons_topic" # 告诉 costmap 去这个话题听形状
        self.info_pub.publish(info_msg)
        self.get_logger().info('已发送 Filter Info (说明书)')

        # 第二步：发送 4 面墙的形状
        for wall in self.walls_polygon:
            poly_stamped_msg = PolygonStamped()
            poly_stamped_msg.header.frame_id = 'map'
            poly_stamped_msg.header.stamp = self.get_clock().now().to_msg()
            poly_stamped_msg.polygon = wall
            self.poly_pub.publish(poly_stamped_msg)
            
        self.get_logger().info('已发送 4 面虚拟围墙！ExploreLite 将被限制在框内。')
        self.timer.cancel() # 发完就销毁定时器

def main(args=None):
    rclpy.init(args=args)
    node = KeepoutBoundaryNode()
    rclpy.spin(node)
    node.destroy_node()
    rclpy.shutdown()


# global_costmap:
#   global_costmap:
#     ros__parameters:
#       update_frequency: 1.0
#       # 注意插件顺序：必须放在 inflation_layer 的前面！
#       plugins: ["static_layer", "obstacle_layer", "keepout_filter", "inflation_layer"]

#       # ... 其他层配置保持不变 ...

#       keepout_filter:
#         plugin: "nav2_costmap_2d::KeepoutFilter"
#         enabled: True
#         # 这里订阅的是“说明书”话题
#         filter_info_topic: "/keepout_filter_info"
